# 0_7 Ideal binary masks versus physical absorbing gratings

**How accurately does an ideal binary amplitude transmission function approximate a physical finite-thickness absorbing grating?**

**Medium and wavelength convention.** The physical comparison uses vacuum wavelength $\lambda_0$, air on both exterior sides ($n_i=n_t=1$), and an independently specified complex absorber index inside the layer. The ideal mask prescribes transmission in that same surrounding medium; it has no independently solved internal material field. The numerical equality $\lambda=\lambda_0$ here follows from the air approximation. See the [course medium convention](0_0_Foundations_of_Scalar_Optics.ipynb#medium-and-wavelength).

## Motivation and prerequisites

**Motivation.** An ideal aperture model is easy to specify and propagate. A material grating can also reflect, absorb and alter phase. We now compare their predicted complex fields and diffraction coefficients under the same illumination, at the same output plane, to identify what the idealization captures and misses.

**Essential prerequisites.** From [0_3](0_3_Boundary_Conditions_and_Diffraction.ipynb), recall a prescribed aperture field; from [0_5](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb), the absorbing structure; from [0_6](0_6_From_Complex_Fields_to_Diffraction_Orders.ipynb), complex coefficients and their reference-plane dependence. You need to distinguish amplitude, phase and intensity. No ability to implement FMM is assumed.

**Optional advanced mathematical background.** Integrating a rectangular transmission gives sinc coefficients. The optional FFT derivation explains normalization and sample origins; it is a check on representation, not a second scattering method.

**First reading.** Begin with the HBVP amplitude and phase, then enable the ideal-mask overlays. Compare one coefficient at a time before changing thickness. This completes the 0 series' conceptual overview of scalar diffraction physics: learn what the plots mean before pursuing every derivation. Subsequent numbered notebooks revisit the individual concepts and mathematics systematically.

**Next lesson.** [Notebook 1: Plane waves and wavevectors](1_0_Plane_Waves_and_Wavevectors.ipynb) returns to a single plane wave and develops its phase, direction and geometry step by step. FMM proficiency is not required.

Run this notebook independently in a fresh kernel. It computes the initial absorbing solution once and reuses it in the thickness sweep. Both models use normal incidence toward $-z$, time factor $e^{-i\omega t}$, and $U_i(0)=1$.


:::{note} Interactive examples
Click the **power button** near the top of the page to activate the examples. Binder supplies Python in your browser; no local installation is needed. Wait for the session to connect and the seven thickness cases to finish computing. Use the two tabs to inspect fields and coefficients. Try **Compare with ideal mask**, **Profile**, **Position x**, **Order m**, **Thickness h**, and **Distance below exit**, then expand the thickness experiment in tab B. Use **Reset absorber** to restore the starting comparison. The text, equations and saved figures can be read without activating a session.
:::

For local use, download the {download}`numerical explorer <downloads/grating_physics_numerical_explorer.zip>` or the complete {download}`worked-example scripts <downloads/grating_physics_scripts.zip>` with their shared helpers.

In [ ]:
from pathlib import Path
import sys
import importlib
from IPython.display import display

_support_dir = next((base for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))

# Release current explorers and any remaining objects from earlier notebook versions.
for _name in ('absorber_comparison', 'binary_masks', 'grating_comparison'):
    if _name in globals():
        globals()[_name].close()
if 'grating_views' in globals():
    for previous in grating_views.values():
        previous.close()

# Refresh dependencies before their consumers, including a cached pre-absorber solver.
# Import and reload one module at a time: importing all first can fail on new symbols.
for _module_name in ('scalar_grating', 'binary_transmission', 'thin_grating',
                     'absorbing_grating', 'scalar_grating_view', 'amplitude_grating',
                     'grating_comparison_view', 'absorbing_grating_view'):
    importlib.reload(importlib.import_module(f'wave_optics.{_module_name}'))

import wave_optics.absorbing_grating as absorbing_model
import wave_optics.scalar_grating_view as grating_view
import wave_optics.amplitude_grating as amplitude_model
import wave_optics.absorbing_grating_view as absorbing_view

# One shared physical absorber: centred air opening, air half-spaces.
# Compute 49 harmonics; display only orders -7,...,+7 in coefficient plots.
grating_config = absorbing_model.AbsorbingGratingConfiguration(M=24)
display_order_limit = 7
grating_solution = absorbing_model.solve_absorbing_grating(grating_config)
grating_views = {}  # Cleanup ownership when switching notebooks in one kernel.


### Which models are being compared?

| Model | Description | Role here |
| --- | --- | --- |
| **A: Ideal binary amplitude mask** | Prescribe transmission 1 in air openings and 0 in opaque regions. | Analytical field and diffraction-coefficient prediction. |
| **B: Physical absorbing grating** | Solve a finite layer of absorbing ridges and air openings, with air on both sides. | Scalar Helmholtz field, reflection, transmission and absorption. |
| **C: Chrome-on-glass grating** | Add actual material data and a glass substrate with additional interfaces. | Future extension; not simulated. |

For A and B, $d=1.50\,\mu\mathrm m$, $\mathrm{CD}=0.75\,\mu\mathrm m$, $D_{\mathrm{open}}=\mathrm{CD}/d=0.5$ and $\lambda_0=0.633\,\mu\mathrm m$. Model B starts at $h=0.30\,\mu\mathrm m$ with index $1.5+0.8i$. These are illustrative absorber parameters, not fitted chrome data. Both exterior regions are air and the opening is centred at $x=0$.

The reference plane must be the same for both predictions. A different plane would add propagation to one field and could masquerade as disagreement between models. Likewise, fitting away a phase offset or renormalizing one field would change the comparison. We retain the shared incident phase and amplitude throughout.

An ideal transmission function is a boundary prescription. It is not a statement that a finite physical material necessarily produces those exact local output values. Comparing the models tests that approximation without changing the scattering calculation.


## 1. How well does the ideal amplitude mask predict the physical field?

We now compare **Model A** with the physical **Model B** introduced in 0_5 and decomposed in 0_6. Both use the same clear width, period, illumination, transverse origin and output plane. The physical model retains finite transmission, reflection, absorption and edge scattering; the ideal mask simply prescribes

$$t_{\mathrm{ideal}}(x)=\begin{cases}1,&|x|<\mathrm{CD}/2,\\0,&\text{otherwise in one period},\end{cases}$$

with periodic repetition. The exact edge convention does not affect its integral coefficients.

**Predict:** for $D_{\mathrm{open}}=1/2$, which nonzero even orders vanish in the ideal mask? Must the absorbing HBVP suppress them too?


### From the prescribed mask to common-plane coefficients

The ideal transmission coefficients are

$$t_m=\frac{1}{d}\int_{-d/2}^{d/2}t_{\mathrm{ideal}}(x)e^{-i2\pi mx/d}\,dx
=D_{\mathrm{open}}\,\operatorname{sinc}(mD_{\mathrm{open}}).$$

A shift of the opening by $x_0$ multiplies this by $e^{-i2\pi mx_0/d}$.

Place the ideal mask at the **exit**, $z=-h$. The shared incident wave has propagated through an air distance $h$, so the prescribed exit field is

$$U_{\mathrm{exit}}^{\mathrm{ideal}}(x)=e^{ik_0h}t_{\mathrm{ideal}}(x),\qquad
C_m^{\mathrm{ideal}}(-h)=e^{ik_0h}t_m.$$

The physical coefficients $C_m^{\mathrm{HBVP}}(-h)$ come from the absorbing solution calculated by this notebook's setup, using the same parameters as 0_5 and 0_6. For either model, at distance $\ell\geq0$ below the exit,

$$z_{\mathrm{ref}}=-h-\ell,\qquad
C_m(z_{\mathrm{ref}})=C_m(-h)e^{i\beta_m\ell},\qquad
\beta_m=\sqrt{k_0^2-(2\pi m/d)^2},$$

using the outgoing or decaying root. **No fitted phase or power normalization is applied.**

**Explore A:** start at the exit. Select real component, imaginary component, amplitude or wrapped phase of the same field; intensity also remains available. Move **Position x** to track one HBVP field value on the spatial profile and the Argand diagram, whose colour indicates $x$. Then move the shared output plane or change thickness. The field plots initially show HBVP alone; enable **Compare with ideal mask** to add the ideal-field overlays.

**Explore B:** compare the direct HBVP coefficients, the FFT of its sampled complex field, and the analytical ideal-mask coefficients. Select $m=1$ to inspect magnitude, phase, the complex-plane point and scalar efficiency. Try $m=2$, which vanishes in the ideal mask at $D_{\mathrm{open}}=1/2$, and $m=3$, which is evanescent here. Expand the thickness experiment to follow the selected order through seven separately solved layers.

The initial thickness reuses the setup solution. The thickness control explores the same absorbing-ridge/air-opening geometry at other layer depths; the separate 0_5 and 0_6 lessons retain their initial thickness. Reset this comparison to return to that shared case.

Here $t_m$ is a coefficient of the prescribed mask transmission. Both $C_m^{\mathrm{ideal}}$ and $C_m^{\mathrm{HBVP}}$ are field coefficients at the indicated plane; their superscripts identify the model. They share the same incident phase reference and the same definition of $U_{\mathrm{exit}}$.

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mn}$) for field coefficients. [Shared convention](0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


In [ ]:
# Reuse the initial absorber solution; solve only the six other thickness cases.
if 'absorber_comparison' in globals():
    absorber_comparison.close()
absorber_cases = amplitude_model.AbsorbingComparisonData(solution=grating_solution)
absorber_comparison = absorbing_view.AbsorbingGratingExplorer(
    absorber_cases, display_order_limit=display_order_limit)
display(absorber_comparison.ui)


### Read the field and coefficient views

At the selected output plane, the same solved HBVP field can be written as

$$\boxed{U=\operatorname{Re}U+i\operatorname{Im}U=|U|e^{i\phi}.}$$

Each coloured **field Argand point** is $U(x,z_{\mathrm{ref}})$ at one transverse position. The position slider highlights that same value on the spatial plots. Different positions can share a complex value; a symmetric field can retrace the curve. This local field point is distinct from an order coefficient $C_m$.

As in 0_6, for period $d$ and $G=2\pi/d$, extract the coefficients from the **complex field**:

$$\boxed{C_m^{\mathrm{HBVP}}(z_{\mathrm{ref}})=\frac1d\int_{-d/2}^{d/2}
U_{\mathrm{HBVP}}(x,z_{\mathrm{ref}})e^{-imGx}\,dx.}$$

View B compares two routes: read $C_m$ directly from the Fourier-modal solution, or apply an FFT to its sampled complex field. The green open squares should coincide with the direct HBVP points. The readout reports the largest absolute complex difference over retained orders, with tolerance **$10^{-12}$** for unit incident amplitude.

This is a **representation-consistency check**, not an independent scattering calculation. Both routes describe the same truncated HBVP solution. Transforming $|U|$ or $|U|^2$ would discard field phase and would not give these diffraction coefficients. Both coefficient magnitudes and phases are needed to reconstruct $U$ and calculate interference in $|U|^2$.

:::{dropdown} Optional: FFT normalization and sample ordering

Use $N$ uniformly spaced samples $x_j=x_0+jd/N$, with the repeated endpoint excluded and $N>2M$ for retained orders $-M,\ldots,M$. Here $x_0=-d/2$ and the default is $N=512$. Then

$$C_m^{\mathrm{FFT}}=\frac{e^{-imGx_0}}{N}\sum_{j=0}^{N-1}U(x_j,z_{\mathrm{ref}})e^{-i2\pi mj/N}.$$

The factor $1/N$ supplies the period-integral normalization; $e^{-imGx_0}$ corrects the sampling origin. Negative orders use the corresponding negative-frequency FFT bins. The complex samples retain the same incident phase and output-plane convention as the direct coefficients and ideal-mask comparison.

:::


### Why does an amplitude structure also change phase?

The ideal mask assigns two field values. The physical field responds to **finite transmission through the absorber, reflection, absorption, edge scattering and repeated interactions within the layer**. Both transmitted amplitude and phase therefore vary. This phase is an outcome of the scattering problem, part of the physical response of the absorbing structure.

- The ideal transmission panel shows the exact binary prescription. With **Compare with ideal mask** enabled, the common-plane profiles show its **49-term Fourier reconstruction** beside the 49-term HBVP field. The coefficient plots display only orders $-7\leq m\leq7$; this does not truncate either field.
- At the exit, the ideal reconstruction rings near jumps; its small ripples in the nominally opaque region can have changing phase. The exact zero field there has no phase. Away from the exit, even the untruncated ideal field diffracts and is no longer binary.

The coefficient view measures

$$\Delta C_m=C_m^{\mathrm{HBVP}}-C_m^{\mathrm{ideal}},\qquad
\Delta\phi_m=\arg\left[C_m^{\mathrm{HBVP}}(C_m^{\mathrm{ideal}})^*\right].$$

A phase is omitted below amplitude $10^{-6}$; the relative phase needs both signals. In air, the transmitted scalar efficiency is

$$\eta_m^{(T)}=\frac{\operatorname{Re}\beta_m}{k_0}|C_m|^2.$$

Propagating orders carry outgoing normal flux; evanescent and grazing orders carry none. Periodicity fixes the allowed directions, while the chosen transmission or scattering model determines the complex coefficients.

For the **physical absorber**, the independent volume-loss integral gives $A$ and verifies $R+T+A=1$. At the initial thickness, $R\approx0.06409$, $T\approx0.42515$ and $A\approx0.51076$. The ideal mask does not predict reflected power or say where rejected energy goes; its missing transmitted flux must not be labelled calculated absorption.

Coefficient phase depends on the reference plane. Both models acquire the same propagation factor for each order, so $\Delta\phi_m$ stays fixed while both signals remain significant. Phases between different orders generally change because their longitudinal wavevectors differ.


### Checkpoint: what does the ideal opening predict?

**Predict:** at $D_{\mathrm{open}}=1/2$, which nonzero even coefficients vanish? Is the phase of a vanishing coefficient meaningful?

**Try it in code:** vary `try_D_open` between `0.25`, `0.50` and `1.00`. This exercise varies the ideal transmission function alone; the physical comparison above retains its own geometry. These are $t_m$, before the exit-plane propagation factor.


In [ ]:
import numpy as np

try_D_open = 0.50  # Clear fraction of one period: 0 <= D_open <= 1.
try_orders = np.arange(-4, 5)
try_coefficients = try_D_open * np.sinc(try_orders * try_D_open)
print("m       |t_m|       arg t_m (rad)")
for try_m, try_c in zip(try_orders, try_coefficients):
    try_phase = f"{np.angle(complex(try_c)):+.3f}" if abs(try_c) >= 1e-6 else "undefined"
    print(f"{try_m:+d}      {abs(try_c):.6f}      {try_phase}")


:::{dropdown} Check your prediction


For $D_{\mathrm{open}}=1/2$, all nonzero even coefficients vanish; tiny computed residues are rounding error. At $D_{\mathrm{open}}=1$, only the zeroth coefficient remains. Negative real coefficients have phase $\pi$, not negative magnitude. These zeros belong to the ideal prescription and need not occur in the physical absorbing-grating solution.


:::


## 2. Does thicker mean closer to ideal?

In View B, expand **Does increasing thickness approach the ideal mask?** Each thickness changes the physical layer and is solved independently. The ideal opening stays the same, but its exit-phase factor $e^{ik_0h}$ changes.

For this sweep the entrance remains at $z=0$ and the comparison plane follows the exit at fixed distance $\ell$: **$z_{\mathrm{ref}}=-h-\ell$ is not one fixed laboratory plane.** Both predictions always use the same plane for each thickness. Absolute phases include propagation; they are not aligned afterward.

- For $m=1$ in these illustrative cases, $|\Delta C_1|$ drops from about $0.227$ at $h=0.03\,\mu\mathrm m$ to $0.081$ at $0.30\,\mu\mathrm m$, then rises to $0.200$ at $1.00\,\mu\mathrm m$.
- Increased opacity removes direct transmission through the absorber, but reflection, edge scattering and phase changes remain.
- An ideal amplitude mask is most useful when those effects are small for the observable of interest; thickness alone is not a guarantee. 

### Checkpoint: is the thickest absorber closest to ideal?

**Predict:** must $|\Delta C_m|$ decrease whenever thickness increases?

**Try it in code:** choose `try_m = 1`, `2` or `3`, then vary `try_distance`. This reads the seven already-computed thickness cases. Each row compares both models at the same plane, $z=-h-\ell$; that plane follows the exit as $h$ changes.


In [ ]:
import numpy as np

try_m = 1
try_distance = 0.0  # ell: micrometres below each case's exit.
try_planes = absorber_cases.planes(try_distance)
try_j = int(np.flatnonzero(absorber_cases.solutions[0].orders == try_m)[0])
print("h (um)     |C physical|     |C ideal|     |difference|")
for try_h, try_plane in zip(absorber_cases.thicknesses, try_planes):
    try_physical = try_plane.hbvp[try_j]
    try_ideal = try_plane.thin[try_j]
    print(f"{try_h:.2f}        {abs(try_physical):.6f}        "
          f"{abs(try_ideal):.6f}      {abs(try_physical - try_ideal):.6f}")


:::{dropdown} Check your prediction


No monotonic approach is guaranteed: thickness changes scattering and phase as well as attenuation. For order 1, the smallest difference among these sampled thicknesses occurs near $h=0.30$ µm; this is not a universal optimum. Moving the plane preserves $|\Delta C_m|$ for a propagating order, but reduces it for an evanescent order because both predictions acquire the same decay factor.


:::


### Complex coefficients and the next physical model

A diffraction coefficient $C_m$ is a complex scattering amplitude for a specified incident field, order and reference plane. Periodicity fixes the allowed transverse wavevectors; the physical model determines their magnitudes and phases.

A chrome-on-glass grating (Model C) adds a substrate and material-specific interfaces. Maxwell's equations also introduce polarization-dependent matching: this scalar $U,\partial_zU$ model coincides with the TE $E_y$ equation for the present nonmagnetic, $y$-invariant geometry, but does not provide both TE and TM responses. Substrate modeling and full vector electromagnetic scattering belong to a later lesson.

- For a technical connection to photomask models and complex scattering coefficients, see Thomas Vincent Pistor, *Electromagnetic Simulation and Modeling with Applications in Lithography* (2001), [official thesis](https://www2.eecs.berkeley.edu/Pubs/TechRpts/2001/Archive/ERL-01-19.pdf).
- Sections 4.3 and 4.7, especially 4.7.2.1–4.7.2.2 and Figures 4-11, 4-14 and 4-15, discuss mask models, transmission fields and reference planes.
- Pistor is a technical reference here, not the originator of scattering coefficients. 

## What have we learned?

An ideal binary amplitude mask prescribes two transmission levels and their Fourier coefficients. A finite-thickness absorbing grating requires a scattering solution: its output generally differs in both amplitude and phase, and its incident power is divided among reflection, transmission and absorption.

**Physical absorbing grating → solved complex field → Fourier decomposition → complex diffraction coefficients.**

The field and its complete complex coefficients carry the same information; compare the physical coefficients with the ideal amplitude-mask prediction at the same output plane.

**Check your understanding:** why is the physical output not exactly binary? Why does moving the output plane change the field but not $R$, $T$ or $A$? Why need a thicker absorber not reproduce the ideal coefficients?

:::{dropdown} Check your answers

Finite transmission, reflection and coupling at the material boundaries shape the field. In the air output region, order phases and evanescent amplitudes evolve while propagating fluxes remain fixed; material absorption already occurred in the layer. Increasing thickness suppresses direct passage through the absorber but also changes scattering and phase. Greater opacity alone does not guarantee the ideal-mask field.

:::

This notebook shares one initial solution with its comparison and solves the six other thicknesses once. The two tabs, output-plane control, selected order and expandable thickness plots stay linked. Reset returns to the initial comparison; rerunning setup releases previous views. Fields and flux totals retain 49 harmonics, while coefficient plots display only orders -7 through +7.

[Previous: 0_6](0_6_From_Complex_Fields_to_Diffraction_Orders.ipynb) · [Back to the chapter index](0_0_Foundations_of_Scalar_Optics.ipynb) · [Continue to Notebook 1: Plane waves and wavevectors](1_0_Plane_Waves_and_Wavevectors.ipynb)


## Worked-out examples

These optional scripts show the numerical implementation of the topics in this notebook.
Use the complete {download}`scripts download <downloads/grating_physics_scripts.zip>` to keep the plotting helper and
`wave_optics` package with the examples. Edit **INPUT PARAMETERS**, then run the file
or execute its `# %%` sections in order. Read **CHECK AND INTERPRET** with the plots.

### 1. From the prescribed mask to common-plane coefficients

**Script:** [0_7_01_ideal_mask_coefficients.py](Wave_solver_py_scripts/scripts/chapter_0/0_7_01_ideal_mask_coefficients.py)

Integrate ideal binary-mask Fourier coefficients.

### 2. Read the field and coefficient views

**Script:** [0_7_02_common_phase_reference.py](Wave_solver_py_scripts/scripts/chapter_0/0_7_02_common_phase_reference.py)

Compare physical and ideal masks with one exit-phase reference.

### 3. Does thicker mean closer to ideal?

**Script:** [0_7_03_absorber_thickness_scan.py](Wave_solver_py_scripts/scripts/chapter_0/0_7_03_absorber_thickness_scan.py)

Test whether greater absorber thickness approaches an ideal mask.

### 4. Complex coefficients and the next physical model

**Script:** [0_7_04_compare_at_common_depth.py](Wave_solver_py_scripts/scripts/chapter_0/0_7_04_compare_at_common_depth.py)

Propagate physical and ideal coefficients to the same observation plane.


## References and further reading

1. **TU Delft, *Interactive Optics*, “Scalar Diffraction Optics.”** [Read online](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html). The rectangular-aperture example explains the sinc spectrum behind the ideal opening's coefficients. Periodic repetition selects discrete spatial harmonics in our model.

2. **Thomas Vincent Pistor (2001), *Electromagnetic Simulation and Modeling with Applications in Lithography*, UC Berkeley dissertation, UCB/ERL M01/19.** [Official thesis](https://www2.eecs.berkeley.edu/Pubs/TechRpts/2001/Archive/ERL-01-19.pdf). Sections 4.7.2.1–4.7.2.2 contrast thin transmission prescriptions with finite-thickness electromagnetic mask models. Read for the modeling distinction; its photomask geometries are not our air/absorber/air example.

3. **NumPy 1.26 manual, “Discrete Fourier Transform.”** [FFT conventions](https://numpy.org/doc/1.26/reference/routines.fft.html). Supports the sampled-field comparison in View B. An FFT of the solved complex field checks its representation; it does not independently solve the scattering problem.


## Selected freely accessible reading

Optional reading: start with the suggested passage. Freely accessible works are not necessarily in the public domain.

- **M. del Mar Sánchez-López, I. Moreno and A. Martínez-García. [Teaching diffraction gratings by means of a phasor analysis](https://opg.optica.org/abstract.cfm?uri=ETOP-2009-EMA1).** ETOP (2009), EMA1; DOI: 10.1364/ETOP.2009.EMA1. **Access:** Publisher marks Open Access; not public domain. **Read for:** Read Sections 1–2: binary gratings, phasors and order amplitudes.

- **V. Liu and S. Fan. [S4: A free electromagnetic solver for layered periodic structures](https://web.stanford.edu/group/fan/publication/Liu_ComputerPhysicsCommunications_183_2233_2012.pdf).** Computer Physics Communications 183, 2233–2244 (2012); DOI: 10.1016/j.cpc.2012.04.026. **Access:** Free author-institution PDF; copyright retained. **Read for:** Optional advanced comparison: vector FMM, not our scalar solver.

**Convention check:** Translate each source's order signs, propagation side, polarization assumptions and amplitude/power normalization to this notebook's stated model.

[Source catalogue and access notes](supplement/reading_sources.json).
